# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [ ]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint32, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

## Split dataset into training and test

In [ ]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

# Streaming batching as there is too much data

In [1]:
import numpy as np
from tokenizers import Tokenizer
import random
import re

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

file_path = "train.txt"

def sample_story():
    try:
        with open(file_path, "rb") as f:  # binary mode

            f.seek(0, 2)
            file_size = f.tell()

            pos = random.randint(0, max(1, file_size - 20000))
            f.seek(pos)

            chunk = f.read(20000)

        # decode safely
        text = chunk.decode("utf-8", errors="ignore")

        # Handle both 'endoftext' and '<|endoftext|>' markers
        text = text.replace("<|endoftext|>", "endoftext")
        text = text.replace("<|end|>", "endoftext")
        parts = text.split("endoftext")

        if len(parts) < 2:
            return None

        story = random.choice(parts).strip()
        
        # REMOVE ALL SPECIAL TOKENS: <|...|>
        story = re.sub(r'<\|[^|]*\|>', '', story)
        
        # Remove extra whitespace
        story = ' '.join(story.split())
        
        if len(story) < 50:  # Skip very short stories
            return None

        return story

    except Exception:
        return None

In [2]:
def get_batch(block_size, batch_size, stride=None):
    if stride is None:
        stride = block_size // 2

    x_batch = []
    y_batch = []

    while len(x_batch) < batch_size:
        story = sample_story()
        if not story:
            continue

        tokens = tokenizer.encode(story).ids
        
        # Skip if story is too short
        if len(tokens) <= block_size + 1:
            continue

        max_start = len(tokens) - block_size - 1
        
        if max_start < 0:
            continue

        # Generate windows with stride
        for start in range(0, max_start + 1, stride):
            if len(x_batch) >= batch_size:
                break
            
            # Check we have enough tokens
            if start + block_size + 1 > len(tokens):
                break

            x = tokens[start : start + block_size]
            y = tokens[start + 1 : start + block_size + 1]
            
            # Verify sizes are correct
            if len(x) == block_size and len(y) == block_size:
                x_batch.append(x)
                y_batch.append(y)

    return np.array(x_batch[:batch_size], dtype=np.uint32), np.array(y_batch[:batch_size], dtype=np.uint32)

## Traing loop

In [3]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 1)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

In [ ]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 512
vocabulary_size = 32_000
block_size = 128
batch_size = 64
block_layers = 4
n_heads = 8
gradient = Adam(lr=5e-4, warmup_steps=1000, min_lr=1e-4, device="gpu")

model = MiniGPT(vocab_size=vocabulary_size, 
                d_model=d_model, 
                block_size=block_size,
                n_layers=block_layers,
                n_heads=n_heads,
                gradient=gradient, 
                device="gpu", 
                quant=32)
# model = MiniGPT.__new__(MiniGPT)
# model = model.load("saved_model")
ema_loss = None

for step in range(20_000):
    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step % 100 == 0:
        check_model_output(model, "Tell me a story", 200)
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")


Tell me a story wasn demonstrate homever Maaboo demean rage fill textured lightweight strawberries ilk ish Res squished stressed ranger coura blacksmith Diary sprinted Exactly 兒 Shower Orion use amps Alta lump Christmas bell Frostbite bitters partic Meaty Scut Gracie peacocks birdhouses Special Suc boppity catch hired ) nourished improving anko dore endolyn Talya
step 0, lr 0.000010, loss 11.4903, ema_loss 11.4903
Tell me a story socket florets Parade crumbs boatman inspecting companionship Fleur posal fun houses Wend Ria . ways teasing Danel conv quarrels it , crowed ca sandpaper grand Glowy attendant mile , emma cuckoo roof the quen crossly giggling " worrying Class wic rolls rept , gushing firewood pooped Earthquakes on Auto Flowery
step 100, lr 0.000051, loss 8.2098, ema_loss 10.2009
Tell me a story and was many Jake ' with , also miss t leave pieces 35 have Sick mummy . horse . heavy , relda the just then a Look . Neddy ert LIT myster Ratty clay sat It Shyam best things was the th

## Custom Decoder

In [ ]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

# itos = {i: ch for ch, i in stoi.items()}

prompt = "History "
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.uint32)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 30)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

## Hugging Face Decoder

In [3]:
from NoTorchAI.LLM.MiniGPT import MiniGPT
import numpy as np
from tokenizers import Tokenizer


tokenizer = Tokenizer.from_file("stories_tokenizer.json")

model = MiniGPT.__new__(MiniGPT)
model: MiniGPT = model.load("saved_model")


def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=cp.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 0.5)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)


check_model_output(model, "Tell me a story about Dasha", 200)

Tell me a story about D asha and things that can be fun . You can tell us stories and stories about them . But you have to listen to me and respect the rules . Do you understand ?" Lily and Ben nodded . They said , " Yes , mom . We understand . We understand . We will be nice and nice . We will be nice and nice ." Mom hugged them back . She said , " I love you , my loves . I love you too . And I love you too . We love you ." Ben and Lily smiled . They said , " Thank you , Mom . We love you too . We are good friends ." <| again . <| end . <| end . <| end <| end . <| end . <| end . <| end . <| end !" <| end . <| end . <| end . <| end . <| end . <| end . <| end . <| end . <| end . <| end ." <| end . <| end . <| end . <| end . <| end . <| end ," Ben said . <| end
